# DeepTrace sprint - WORKER B
Runtime **T4 GPU**. Two copies of this notebook run in parallel (worker A and worker B) and share work through Drive.
**Runtime -> Run all.** If it disconnects, Run all again: extraction, training and evaluation all resume.
Worker A: Celeb-DF extraction + first 800 FF++ videos, then training queue A + mouth (E2) evaluation + timing.
Worker B: remaining FF++ videos, then training queue B + degradation evaluation.

In [1]:
# Cell 1 - mount Drive and install the deeptrace package from this notebook's embedded copy
from google.colab import drive
drive.mount("/content/drive")
import base64, io, tarfile, sys, os, hashlib
PKG_B64 = "<embedded deeptrace package omitted - identical to the deeptrace/ folder in this repo>"
CODE = "/content/drive/MyDrive/DeepTrace/code"
os.makedirs(CODE, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PKG_B64)), mode="r:gz") as t:
    t.extractall(CODE)
sys.path.insert(0, CODE)
print("package version", hashlib.md5(PKG_B64.encode()).hexdigest()[:10], "->", CODE)


Mounted at /content/drive


/tmp/ipykernel_1551/1045401993.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(CODE)


package version 41f85a5738 -> /content/drive/MyDrive/DeepTrace/code


In [2]:
# Cell 2 - setup: installs, credentials, datasets, splits
WORKER = "B"
import subprocess, sys, os, json, time, torch, pandas as pd
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch"], check=True)
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
import kagglehub
from deeptrace import paths
from deeptrace.data import prep
from deeptrace import train as T, experiments as X
prep.TAG = WORKER
inv = pd.read_csv(paths.MANIFESTS / "inventory.csv")
ff_rows = T.load_split("ffpp").sort_values("rel").reset_index(drop=True)          # 5,000 videos, 4 methods
cdf_rows = inv[inv.dataset == "celebdf"].sort_values("rel").reset_index(drop=True)
t0 = time.time()
CDF = next(p.parent for p in Path(kagglehub.dataset_download("reubensuju/celeb-df-v2")).rglob("List_of_testing_videos.txt"))
FF = Path(kagglehub.dataset_download("xdxd003/ff-c23"))
print(f"worker {WORKER} | data ready in {time.time()-t0:.0f}s | cpus {os.cpu_count()} | {torch.cuda.get_device_name(0)}")
print("ffpp videos:", len(ff_rows), "| celebdf videos:", len(cdf_rows))


Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'ff-c23' dataset.
worker B | data ready in 3s | cpus 2 | Tesla T4
ffpp videos: 5000 | celebdf videos: 6529


In [3]:
# Cell 3 - extraction share of worker B (resumable)
prep.run(ff_rows.iloc[800:], "ffpp", FF)


[ffpp] done 24, to do 4184
  400/4184  0.67 vid/s  eta 95 min
  800/4184  0.67 vid/s  eta 85 min
  1200/4184  0.66 vid/s  eta 76 min
  1600/4184  0.67 vid/s  eta 64 min
  2000/4184  0.66 vid/s  eta 55 min
  2400/4184  0.66 vid/s  eta 45 min
  2800/4184  0.66 vid/s  eta 35 min
  3200/4184  0.66 vid/s  eta 25 min
  3600/4184  0.67 vid/s  eta 15 min
  4000/4184  0.66 vid/s  eta 5 min
[ffpp] finished 4184 in 105.6 min


In [4]:
# Cell 4 - wait until BOTH workers have finished extracting (pulls the other worker's shards from Drive)
need = {"celebdf": set(cdf_rows.rel), "ffpp": set(ff_rows.rel)}
while True:
    left = {ds: len(need[ds] - prep.restore(ds)) for ds in need}
    print(time.strftime("%H:%M:%S"), "videos still being extracted:", left, flush=True)
    if all(v == 0 for v in left.values()):
        break
    time.sleep(60)
print("ALL CROPS AVAILABLE")


16:45:36 videos still being extracted: {'celebdf': 0, 'ffpp': 0}
ALL CROPS AVAILABLE


In [5]:
# Cell 5 - training queue B (finished runs are skipped)
ROBUST = dict(p_jpeg=0.5, jpeg_min=20, p_blur=0.3, p_down=0.4)
Q = [dict(name="v1_mean",    train_on="celebdf_strict", head="mean"),
     dict(name="v2_tdiff",   train_on="ffpp", head="tdiff"),
     dict(name="robust",     train_on="ffpp", head="mean", aug=ROBUST),
     dict(name="lomo_noNT",  train_on="ffpp", head="mean", methods=["Deepfakes", "Face2Face", "FaceSwap"]),
     dict(name="v2_bilstm",  train_on="celebdf_strict", head="bilstm"),
     dict(name="v1_mean",    train_on="celebdf_strict", head="mean", seed=1),
     dict(name="v1_mean",    train_on="celebdf_strict", head="mean", seed=2)]
for cfg in Q:
    try:
        T.run(cfg)
    except Exception as e:
        print("RUN FAILED", cfg, repr(e))
print("QUEUE B DONE")


[v1_mean_celebdf_strict_s0] train 2603 (2051 fake) | val 220 (101 fake)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  ep1/8 loss 0.1606 val_auc 0.9795 (87s)
  ep2/8 loss 0.0738 val_auc 0.9726 (78s)
  ep3/8 loss 0.0475 val_auc 0.9880 (79s)
  ep4/8 loss 0.0462 val_auc 0.9864 (78s)
  ep5/8 loss 0.0275 val_auc 0.9946 (77s)
  ep6/8 loss 0.0137 val_auc 0.9955 (76s)
  ep7/8 loss 0.0161 val_auc 0.9950 (76s)
  ep8/8 loss 0.0085 val_auc 0.9953 (78s)
  [data] 1 videos have no face crops and are excluded: {('test', 0): 1}
  test celebdf_strict    AUC 0.9954  bal_acc 0.962  real_rec 0.945 fake_rec 0.980
  test ffpp              AUC 0.6710  bal_acc 0.566  real_rec 0.979 fake_rec 0.154
[v2_tdiff_ffpp_s0] train 3600 (2880 fake) | val 700 (560 fake)
  ep1/8 loss 0.2764 val_auc 0.8765 (126s)
  ep2/8 loss 0.2246 val_auc 0.9146 (122s)
  ep3/8 loss 0.1813 val_auc 0.9570 (121s)
  ep4/8 loss 0.1523 val_auc 0.9665 (120s)
  ep5/8 loss 0.1220 val_auc 0.9715 (112s)
  ep6/8 loss 0.0978 val_auc 0.9760 (114s)
  ep7/8 loss 0.0851 val_auc 0.9751 (111s)
  ep8/8 loss 0.0793 val_auc 0.9779 (113s)
  test ffpp              AUC 0.9691  

In [6]:
def wait_for(run_id):
    while not (paths.RUNS / run_id / "metrics.json").exists():
        print(time.strftime("%H:%M:%S"), "waiting for", run_id, flush=True); time.sleep(60)
# Cell 6 - degradation evaluation (G5), crop level, threshold fixed from clean validation
for rid in ["v1_mean_celebdf_strict_s0", "v1_mean_ffpp_s0", "robust_ffpp_s0", "v2_bilstm_ffpp_s0",
            "v2_tdiff_ffpp_s0", "v3_freq_ffpp_s0"]:
    wait_for(rid)
    if not (paths.RUNS / rid / "degradation.csv").exists():
        X.degradation(rid)
print("EVAL B DONE")


  [data] 1 videos have no face crops and are excluded: {('test', 0): 1}


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


v1_mean_celebdf_strict_s0 celebdf_strict clean 0.9954
v1_mean_celebdf_strict_s0 celebdf_strict jpeg50 0.9914
v1_mean_celebdf_strict_s0 celebdf_strict jpeg30 0.9822
v1_mean_celebdf_strict_s0 celebdf_strict down112 0.9928
v1_mean_celebdf_strict_s0 celebdf_strict down64 0.9816
v1_mean_celebdf_strict_s0 celebdf_strict jpeg50+down112 0.9844
v1_mean_celebdf_strict_s0 ffpp clean 0.6711
v1_mean_celebdf_strict_s0 ffpp jpeg50 0.6508
v1_mean_celebdf_strict_s0 ffpp jpeg30 0.6347
v1_mean_celebdf_strict_s0 ffpp down112 0.6476
v1_mean_celebdf_strict_s0 ffpp down64 0.6066
v1_mean_celebdf_strict_s0 ffpp jpeg50+down112 0.6298
v1_mean_ffpp_s0 ffpp clean 0.9737
v1_mean_ffpp_s0 ffpp jpeg50 0.9431
v1_mean_ffpp_s0 ffpp jpeg30 0.9067
v1_mean_ffpp_s0 ffpp down112 0.9285
v1_mean_ffpp_s0 ffpp down64 0.8104
v1_mean_ffpp_s0 ffpp jpeg50+down112 0.8888
  [data] 1 videos have no face crops and are excluded: {('test', 0): 1}
v1_mean_ffpp_s0 celebdf_strict clean 0.8689
v1_mean_ffpp_s0 celebdf_strict jpeg50 0.827
v1_mea

In [7]:
# Cell 7 - summary of everything finished so far (safe to run any time)
tab = T.collect()
tab = tab[~tab.name.str.startswith("smoke")]
print(tab.groupby(["name", "train_on", "test"]).agg(auc=("auc", "mean"), auc_std=("auc", "std"), n=("seed", "nunique"),
      bal_acc=("bal_acc", "mean"), real_rec=("real_rec", "mean"), fake_rec=("fake_rec", "mean")).round(4).to_string())
tab.to_csv(paths.REPORTS / "all_runs.csv", index=False)
pm = {f.parent.name: json.loads(f.read_text())["ffpp"].get("per_method_auc", {}) for f in sorted(paths.RUNS.glob("*_ffpp_s*/metrics.json"))}
print("\nFF++ test AUC per method:"); print(pd.DataFrame(pm).T.round(4).to_string())
deg = [pd.read_csv(f) for f in paths.RUNS.glob("*/degradation.csv")]
if deg:
    deg = pd.concat(deg); deg.to_csv(paths.REPORTS / "degradation_all.csv", index=False)
    print("\nDegradation AUC:"); print(deg.pivot_table(index=["run", "test"], columns="cond", values="auc").round(4).to_string())
mo = [pd.read_csv(f) for f in paths.RUNS.glob("*/mouth_eval_*.csv")]
if mo:
    mo = pd.concat(mo); mo.to_csv(paths.REPORTS / "mouth_eval_all.csv", index=False)
    print("\nMouth evaluation:"); print(mo.round(4).to_string())


                                              auc  auc_std  n  bal_acc  real_rec  fake_rec
name      train_on       test                                                             
e2_masked ffpp           celebdf_official  0.6931      NaN  1   0.6172    0.7697    0.4647
                         celebdf_strict    0.7662      NaN  1   0.6821    0.7523    0.6120
                         ffpp              0.9085      NaN  1   0.8125    0.8571    0.7679
lomo_noNT ffpp           celebdf_official  0.7951      NaN  1   0.5799    0.9775    0.1824
                         celebdf_strict    0.8547      NaN  1   0.6675    0.9771    0.3580
                         ffpp              0.9310      NaN  1   0.8545    0.9571    0.7518
robust    ffpp           celebdf_official  0.8441      NaN  1   0.7398    0.8090    0.6706
                         celebdf_strict    0.8811      NaN  1   0.7890    0.7661    0.8120
                         ffpp              0.9606      NaN  1   0.9062    0.9143    0.8982

KeyError: 'ffpp'